# Regression Week 2: Multiple Regression (Interpretation)

The goal of this first notebook is to explore multiple regression and feature engineering with pandas and scikit-learn.

In this notebook you will use data on house sales in King County to predict prices using multiple regression. You will:
* Use pandas DataFrames to do some feature engineering
* Use scikit-learn functions to compute the regression weights (coefficients/parameters)
* Given the regression weights, predictors and outcome write a function to compute the Residual Sum of Squares
* Look at coefficients and interpret their meanings
* Evaluate multiple models via RSS

# Import relevant libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Load in house sales data

Dataset is from house sales in King County, the region where the city of Seattle, WA is located.

In [2]:
sales = pd.read_csv('kc_house_data.csv')

# Split data into training and testing.
We use seed=0 so that everyone running this notebook gets the same results.  In practice, you may set a random seed (or let scikit-learn pick a random seed for you).

In [3]:
train_data, test_data = train_test_split(sales, test_size=0.2, random_state=0)

# Create copies to avoid SettingWithCopyWarning warnings during feature engineering
train_data = train_data.copy()
test_data = test_data.copy()

# Learning a multiple regression model

Recall we can use the following code to learn a multiple regression model predicting 'price' based on the following features:
example_features = ['sqft_living', 'bedrooms', 'bathrooms'] on training data with the following code:

In [4]:
example_features = ['sqft_living', 'bedrooms', 'bathrooms']
example_model = LinearRegression()
example_model.fit(train_data[example_features], train_data['price'])

LinearRegression()

Now that we have fitted the model we can extract the regression weights (coefficients) as a pandas DataFrame as follows:

In [5]:
example_weight_summary = pd.DataFrame({
    'name': ['intercept'] + example_features,
    'value': [example_model.intercept_] + list(example_model.coef_)
})
print(example_weight_summary)

          name         value
0    intercept  67512.015138
1  sqft_living    313.170550
2     bedrooms -56754.666514
3    bathrooms   6887.719108


# Making Predictions

In the gradient descent notebook we use numpy to do our regression. In this book we will use scikit-learn functions to analyze multiple regressions. 

Recall that once a model is built we can use the .predict() function to find the predicted values for data we pass. For example using the example model above:

In [ ]:
example_predictions = example_model.predict(train_data[example_features])
print(example_predictions[0])

395813.49880289414


# Compute RSS

Now that we can make predictions given the model, let's write a function to compute the RSS of the model. Complete the function below to calculate RSS given the model, data, and the outcome.

In [8]:
def get_residual_sum_of_squares(model, data, outcome):
    # First get the predictions
    predictions = model.predict(data)
    # Then compute the residuals/errors
    errors = outcome - predictions
    # Then square and add them up
    RSS = (errors ** 2).sum()
    return(RSS)    

Test your function by computing the RSS on TEST data for the example model:

In [ ]:
rss_example_train = get_residual_sum_of_squares(example_model, test_data[example_features], test_data['price'])
print(rss_example_train)

259213572106085.34


# Create some new features

Although we often think of multiple regression as including multiple different features (e.g. # of bedrooms, squarefeet, and # of bathrooms) but we can also consider transformations of existing features e.g. the log of the squarefeet or even "interaction" features such as the product of bedrooms and bathrooms.

You will use the logarithm function to create a new feature. so first you should import it from the math library.

In [10]:
from math import log

Next create the following 4 new features as column in both TEST and TRAIN data:
* bedrooms_squared = bedrooms\*bedrooms
* bed_bath_rooms = bedrooms\*bathrooms
* log_sqft_living = log(sqft_living)
* lat_plus_long = lat + long 
As an example here's the first one:

In [11]:
train_data['bedrooms_squared'] = train_data['bedrooms'].apply(lambda x: x**2)
test_data['bedrooms_squared'] = test_data['bedrooms'].apply(lambda x: x**2)

In [12]:
# create the remaining 3 features in both TEST and TRAIN data
train_data['bed_bath_rooms'] = train_data['bedrooms'] * train_data['bathrooms']
test_data['bed_bath_rooms'] = test_data['bedrooms'] * test_data['bathrooms']

train_data['log_sqft_living'] = train_data['sqft_living'].apply(lambda x: log(x))
test_data['log_sqft_living'] = test_data['sqft_living'].apply(lambda x: log(x))

train_data['lat_plus_long'] = train_data['lat'] + train_data['long']
test_data['lat_plus_long'] = test_data['lat'] + test_data['long']


* Squaring bedrooms will increase the separation between not many bedrooms (e.g. 1) and lots of bedrooms (e.g. 4) since 1^2 = 1 but 4^2 = 16. Consequently this feature will mostly affect houses with many bedrooms.
* bedrooms times bathrooms gives what's called an "interaction" feature. It is large when *both* of them are large.
* Taking the log of squarefeet has the effect of bringing large values closer together and spreading out small values.
* Adding latitude to longitude is totally non-sensical but we will do it anyway (you'll see why)

**Quiz Question: What is the mean (arithmetic average) value of your 4 new features on TEST data? (round to 2 digits)**

# Learning Multiple Models

Now we will learn the weights for three (nested) models for predicting house prices. The first model will have the fewest features the second model will add one more feature and the third will add a few more:
* Model 1: squarefeet, # bedrooms, # bathrooms, latitude & longitude
* Model 2: add bedrooms\*bathrooms
* Model 3: Add log squarefeet, bedrooms squared, and the (nonsensical) latitude + longitude

In [13]:
model_1_features = ['sqft_living', 'bedrooms', 'bathrooms', 'lat', 'long']
model_2_features = model_1_features + ['bed_bath_rooms']
model_3_features = model_2_features + ['bedrooms_squared', 'log_sqft_living', 'lat_plus_long']

Now that you have the features, learn the weights for the three different models for predicting target = 'price' using LinearRegression() and look at the value of the weights/coefficients:

In [14]:
# Learn the three models:
# model_1 = LinearRegression().fit(...)
model_1 = LinearRegression().fit(train_data[model_1_features], train_data['price'])
model_2 = LinearRegression().fit(train_data[model_2_features], train_data['price'])
model_3 = LinearRegression().fit(train_data[model_3_features], train_data['price'])


In [15]:
# Examine/extract each model's coefficients:
model_1_coefficients = pd.DataFrame({
    'name': ['intercept'] + model_1_features,
    'value': [model_1.intercept_] + list(model_1.coef_)
})
print(model_1_coefficients)


          name         value
0    intercept -7.087085e+07
1  sqft_living  3.129420e+02
2     bedrooms -5.309627e+04
3    bathrooms  1.477704e+04
4          lat  6.539833e+05
5         long -3.257073e+05


In [16]:
model_2_coefficients = pd.DataFrame({
    'name': ['intercept'] + model_2_features,
    'value': [model_2.intercept_] + list(model_2.coef_) 
})
print(model_2_coefficients)

             name         value
0       intercept -6.860682e+07
1     sqft_living  3.068196e+02
2        bedrooms -1.046047e+05
3       bathrooms -7.018153e+04
4             lat  6.505910e+05
5            long -3.099658e+05
6  bed_bath_rooms  2.494415e+04


In [17]:
model_3_coefficients = pd.DataFrame({
    'name': ['intercept'] + model_3_features,
    'value': [model_3.intercept_] + list(model_3.coef_)
})
print(model_3_coefficients)

               name         value
0         intercept -6.262845e+07
1       sqft_living  5.378081e+02
2          bedrooms  2.780479e+03
3         bathrooms  1.013638e+05
4               lat  5.307984e+05
5              long -4.096554e+05
6    bed_bath_rooms -1.818226e+04
7  bedrooms_squared  7.245799e+02
8   log_sqft_living -5.710300e+05
9     lat_plus_long  1.211430e+05


**Quiz Question: What is the sign (positive or negative) for the coefficient/weight for 'bathrooms' in model 1?**

**Quiz Question: What is the sign (positive or negative) for the coefficient/weight for 'bathrooms' in model 2?**

Think about what this means.

# Comparing multiple models

Now that you've learned three models and extracted the model weights we want to evaluate which model is best.

First use your functions from earlier to compute the RSS on TRAINING Data for each of the three models.

In [18]:
# Compute the RSS on TRAINING data for each of the three models and record the values:
rss_model_1_train = get_residual_sum_of_squares(model_1, train_data[model_1_features], train_data['price'])
rss_model_2_train = get_residual_sum_of_squares(model_2, train_data[model_2_features], train_data['price'])
rss_model_3_train = get_residual_sum_of_squares(model_3, train_data[model_3_features], train_data['price'])

print(rss_model_1_train)
print(rss_model_2_train)
print(rss_model_3_train)

979843597588329.0
970799199729578.2
913653644974958.2


**Quiz Question: Which model (1, 2 or 3) has lowest RSS on TRAINING Data?** Is this what you expected?

Now compute the RSS on on TEST data for each of the three models.

In [20]:
# Compute the RSS on TESTING data for each of the three models and record the values:
rss_model_1_test = get_residual_sum_of_squares(model_1, test_data[model_1_features], test_data['price'])
rss_model_2_test = get_residual_sum_of_squares(model_2, test_data[model_2_features], test_data['price'])
rss_model_3_test = get_residual_sum_of_squares(model_3, test_data[model_3_features], test_data['price'])

print(rss_model_1_test)
print(rss_model_2_test)
print(rss_model_3_test)


213487129319107.0
210778544168943.9
203972051917617.1


**Quiz Question: Which model (1, 2 or 3) has lowest RSS on TESTING Data?** Is this what you expected? Think about the features that were added to each model from the previous.